<a id="title"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:150%; text-align:center; border-radius:20px 20px;">**Phase 3.2 — Segment-wise Stenosis Aggregation**</p>

## Method Overview

This notebook implements the core clinical summarization step of **Block 3**: it transforms the point-wise global coronary tree dataset into a **segment-level stenosis table**.

The input is the label-enriched global table produced by Block 3 phase 1, normally located at:

`results/block3_results/label/{sample_name}/total_df_{sample_name}.xlsx`

For each anatomical segment, the notebook keeps the **maximum point-wise % area stenosis (`pct_AS`)** as the conservative segment severity estimate and assigns a **segment-level stenosis severity grade (0-5)** using the stenosis ranges described in CAD-RADS 2.0.

Important: this is **not yet the final patient-level CAD-RADS score**. The patient score will be computed later from the segment summaries and any additional CAD-RADS modifiers/rules. This notebook only prepares the per-segment anatomical summary needed for that future step.

The default sample is `Normal_1`, but all paths are parameterized with `SAMPLE_NAME` so the workflow can be rerun for any ASOCA case.

<a id="step1"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">1. Imports & Configuration</p>

This section sets the target sample and resolves the repository root. Change only `SAMPLE_NAME` to rerun the notebook for another patient, for example `Normal_6` or `Diseased_1`.

In [1]:
# Purpose: import dependencies and configure the sample-level segment stenosis workflow.
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

SAMPLE_NAME = "Normal_1"  # Change this value to process another sample.

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError(
            "Could not locate the project root containing src/. "
            "Run Jupyter with the working directory inside the repository."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

BLOCK3_LABEL_ROOT = PROJECT_ROOT / "results" / "block3_results" / "label"
BLOCK3_LABELS_ROOT = PROJECT_ROOT / "results" / "block3_results" / "labels"  # fallback spelling
BLOCK2_STENOSIS_ROOT = PROJECT_ROOT / "results" / "block2_results" / "stenosis"
OUTPUT_ROOT = PROJECT_ROOT / "results" / "block3_results" / "segment stenosis"

print(f"Sample name  : {SAMPLE_NAME}")
print(f"Project root : {PROJECT_ROOT}")
print(f"Output root  : {OUTPUT_ROOT}")

Sample name  : Normal_1
Project root : C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository
Output root  : C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\results\block3_results\segment stenosis


<a id="step2"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">2. Load Label-Enriched Global Tree</p>

The expected input is the **single merged/global tree** exported by Block 3 phase 1. This table already carries the anatomical `Segment_ID` inherited from the ASOCA label volume and the point-wise stenosis metric from Block 2.

The loader checks the current Block 3 path first, then falls back to the plural `labels` folder, the original Block 2 stenosis export, and the older root-level experimental export.

In [10]:
# Purpose: load the global point-wise table produced by Block 3 label enrichment.
def resolve_global_tree_path(sample_name: str) -> Path:
    """Resolve the global tree spreadsheet for the selected sample."""
    candidates = [
        BLOCK3_LABEL_ROOT / sample_name / f"total_df_{sample_name}.xlsx",
        BLOCK3_LABEL_ROOT / sample_name / f"total_df_merged_{sample_name}.xlsx",
        BLOCK3_LABELS_ROOT / sample_name / f"total_df_{sample_name}.xlsx",
        BLOCK3_LABELS_ROOT / sample_name / f"total_df_merged_{sample_name}.xlsx",
        BLOCK2_STENOSIS_ROOT / sample_name / f"total_df_{sample_name}.xlsx",
        PROJECT_ROOT / f"total_df_{sample_name}.xlsx",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    searched = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(
        "Could not find the global tree spreadsheet for "
        f"{sample_name}. Checked:\n{searched}"
    )


def resolve_pct_as_column(df: pd.DataFrame) -> str:
    """Find the point-wise percent area stenosis column using known project aliases."""
    aliases = [
        "pct_AS",
        "pct_as",
        "Percent_AS",
        "percent_AS",
        "%AS",
        "AS_percent",
        "Area_Stenosis_Percent",
    ]
    for alias in aliases:
        if alias in df.columns:
            return alias
    raise KeyError(
        "No percent area stenosis column found. Expected one of: "
        + ", ".join(aliases)
    )


GLOBAL_TREE_PATH = resolve_global_tree_path(SAMPLE_NAME)
total_df_merged = pd.read_excel(GLOBAL_TREE_PATH)

if "Segment_ID" not in total_df_merged.columns:
    raise KeyError("Input table must contain a Segment_ID column from Block 3 labeling.")

PCT_AS_COL = resolve_pct_as_column(total_df_merged)
total_df_merged["Segment_ID"] = pd.to_numeric(total_df_merged["Segment_ID"], errors="coerce").fillna(0).astype(int)
total_df_merged[PCT_AS_COL] = pd.to_numeric(total_df_merged[PCT_AS_COL], errors="coerce")

print(f"Loaded file        : {GLOBAL_TREE_PATH}")
print(f"Rows / columns     : {total_df_merged.shape[0]} / {total_df_merged.shape[1]}")
print(f"Stenosis column    : {PCT_AS_COL}")
print(f"Segment_ID values  : {sorted(total_df_merged['Segment_ID'].dropna().unique().tolist())}")
display(total_df_merged.head())

Loaded file        : C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\results\block3_results\label\Normal_2\total_df_Normal_2.xlsx
Rows / columns     : 3835 / 16
Stenosis column    : pct_AS
Segment_ID values  : [0, 1, 2, 3, 5, 6, 7, 8, 10, 11, 13, 17, 19]


,Sample_ID,Artery_Type,Branch_ID,Path_Point_Index,Px,Py,Pz,Radius,PointType,Segment_ID,Area,gd,Area_prox,Area_dist,A_ref,pct_AS
0,2,RCA,RCA_B03,61,173.698502,254.093597,-89.938438,0.731369,Standard,19,1.810542,11.238445,34.191562,2.543958,18.36776,90.142826
1,2,RCA,RCA_B03,63,173.506653,253.995590,-89.787567,0.743639,Standard,19,1.829813,11.501634,34.191562,2.485678,18.33862,90.022080
2,2,RCA,RCA_B03,62,173.541321,254.009201,-89.813065,0.740039,Standard,19,1.829813,11.456498,34.191562,2.485678,18.33862,90.022080
3,2,RCA,RCA_B03,60,173.818375,254.207535,-90.083595,0.748802,Standard,19,1.910386,11.018396,34.191562,2.543958,18.36776,89.599242
4,2,RCA,RCA_B03,59,173.876419,254.225555,-90.133186,0.749388,Standard,19,1.970595,10.939954,34.191562,2.523018,18.35729,89.265328


<a id="step3"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">3. Scientific Segment Dictionary</p>

The dictionary below maps ASOCA/AHA segment labels to clinically interpretable coronary territories. The `Segment_ID` values are the labels carried by the global tree dataset.

To preserve the project workflow, the artery information is stored at two levels:

- **`Artery_Type`:** the high-level tree used throughout the pipeline (`RCA` or `LCA`).
- **`Specific_Artery`:** the anatomical artery/branch territory (`RCA`, `LCA`, `LAD`, `LCX`, etc.).

Clinical interpretation notes:

- **Left Main (LM, segment 5):** supplies a large fraction of the left ventricular myocardium through LAD and LCX flow. A high-grade stenosis here is especially critical because it can compromise multiple downstream territories at once.
- **Proximal LAD (segment 6):** often called a high-risk location because it supplies a broad anterior/septal territory before major diagonal branches arise.
- **RCA proximal/mid/distal and PDA (segments 1-4):** summarize the right coronary supply, including inferior wall perfusion when the PDA arises from the RCA.
- **LCX and obtuse marginal/PDA/posterolateral branches (segments 11-13, 15, 17):** capture lateral and inferolateral territories, with clinical importance depending on dominance and downstream myocardial supply.
- **Diagonal branches (segments 9-10):** branch-vessel disease may be smaller in territory than LM/proximal LAD disease, but can still be clinically relevant when stenosis is severe or the branch is large.

In [11]:
# Purpose: define the ASOCA/AHA coronary segment dictionary used for aggregation.
segment_dictionary = pd.DataFrame(
    [
        {"Segment_ID": 1, "Segment_Name": "Proximal RCA", "Artery_Type": "RCA", "Specific_Artery": "RCA"},
        {"Segment_ID": 2, "Segment_Name": "Mid RCA", "Artery_Type": "RCA", "Specific_Artery": "RCA"},
        {"Segment_ID": 3, "Segment_Name": "Distal RCA", "Artery_Type": "RCA", "Specific_Artery": "RCA"},
        {"Segment_ID": 4, "Segment_Name": "Right PDA", "Artery_Type": "RCA", "Specific_Artery": "RCA"},
        {"Segment_ID": 5, "Segment_Name": "Left Main", "Artery_Type": "LCA", "Specific_Artery": "LCA"},
        {"Segment_ID": 6, "Segment_Name": "Proximal LAD", "Artery_Type": "LCA", "Specific_Artery": "LAD"},
        {"Segment_ID": 7, "Segment_Name": "Mid LAD", "Artery_Type": "LCA", "Specific_Artery": "LAD"},
        {"Segment_ID": 8, "Segment_Name": "Distal LAD", "Artery_Type": "LCA", "Specific_Artery": "LAD"},
        {"Segment_ID": 9, "Segment_Name": "First diagonal (D1)", "Artery_Type": "LCA", "Specific_Artery": "LAD"},
        {"Segment_ID": 10, "Segment_Name": "Second diagonal (D2)", "Artery_Type": "LCA", "Specific_Artery": "LAD"},
        {"Segment_ID": 11, "Segment_Name": "Proximal LCX", "Artery_Type": "LCA", "Specific_Artery": "LCX"},
        {"Segment_ID": 12, "Segment_Name": "First obtuse marginal (OM1)", "Artery_Type": "LCA", "Specific_Artery": "LCX"},
        {"Segment_ID": 13, "Segment_Name": "Distal LCX", "Artery_Type": "LCA", "Specific_Artery": "LCX"},
        {"Segment_ID": 15, "Segment_Name": "Left coronary PDA", "Artery_Type": "LCA", "Specific_Artery": "LCX"},
        {"Segment_ID": 17, "Segment_Name": "LCX posterolateral branch", "Artery_Type": "LCA", "Specific_Artery": "LCX"},
    ]
).sort_values("Segment_ID")

present_segments = set(total_df_merged.loc[total_df_merged["Segment_ID"] != 0, "Segment_ID"].unique())
defined_segments = set(segment_dictionary["Segment_ID"].unique())
unmapped_present_segments = sorted(present_segments - defined_segments)

print("Defined segment dictionary:")
display(segment_dictionary)

if unmapped_present_segments:
    print(
        "Warning: these non-background Segment_ID values are present in the data "
        f"but not in the current dictionary: {unmapped_present_segments}"
    )

Defined segment dictionary:


,Segment_ID,Segment_Name,Artery_Type,Specific_Artery
0,1,Proximal RCA,RCA,RCA
1,2,Mid RCA,RCA,RCA
2,3,Distal RCA,RCA,RCA
3,4,Right PDA,RCA,RCA
4,5,Left Main,LCA,LCA
5,6,Proximal LAD,LCA,LAD
6,7,Mid LAD,LCA,LAD
7,8,Distal LAD,LCA,LAD
8,9,First diagonal (D1),LCA,LAD
9,10,Second diagonal (D2),LCA,LAD


<a id="step4"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">4. Segment-wise Aggregation Logic</p>

The point-wise tree can contain thousands of rows per patient. This phase distills those rows into the **maximum stenosis severity observed within each anatomical segment**, which is the segment evidence needed before any patient-level CAD-RADS scoring is attempted.

Implementation rule:

1. Remove `Segment_ID == 0` because it represents background/unassigned points.
2. Group by `Segment_ID`.
3. Compute the maximum point-wise `%AS` within each segment.
4. Count total points and valid stenosis points to document how much evidence supports each segment summary.

In [12]:
# Purpose: aggregate point-wise stenosis into one row per anatomical segment.
foreground_df = total_df_merged.loc[total_df_merged["Segment_ID"] != 0].copy()

segment_aggregation = (
    foreground_df.groupby("Segment_ID", as_index=False)
    .agg(
        Max_pct_AS=(PCT_AS_COL, "max"),
        Mean_pct_AS=(PCT_AS_COL, "mean"),
        Point_Count=(PCT_AS_COL, "size"),
        Valid_pct_AS_Count=(PCT_AS_COL, "count"),
    )
    .sort_values("Segment_ID")
)

segment_summary = segment_aggregation.merge(segment_dictionary, on="Segment_ID", how="left")
segment_summary["Segment_Name"] = segment_summary["Segment_Name"].fillna(
    "Unmapped segment " + segment_summary["Segment_ID"].astype(str)
)
segment_summary["Artery_Type"] = segment_summary["Artery_Type"].fillna("Unmapped")
segment_summary["Specific_Artery"] = segment_summary["Specific_Artery"].fillna("Unmapped")

segment_summary = segment_summary[
    [
        "Segment_ID",
        "Segment_Name",
        "Artery_Type",
        "Specific_Artery",
        "Max_pct_AS",
        "Mean_pct_AS",
        "Point_Count",
        "Valid_pct_AS_Count",
    ]
]

print(f"Foreground rows used for aggregation : {len(foreground_df)}")
print(f"Segments summarized                  : {len(segment_summary)}")
display(segment_summary)

Foreground rows used for aggregation : 3832
Segments summarized                  : 12


,Segment_ID,Segment_Name,Artery_Type,Specific_Artery,Max_pct_AS,Mean_pct_AS,Point_Count,Valid_pct_AS_Count
0,1,Proximal RCA,RCA,RCA,71.371532,25.231111,129,60
1,2,Mid RCA,RCA,RCA,59.746584,0.642120,349,349
2,3,Distal RCA,RCA,RCA,54.120927,-7.990434,411,286
3,5,Left Main,LCA,LCA,-82.557415,-167.663926,80,26
4,6,Proximal LAD,LCA,LAD,83.205781,-40.626722,129,129
5,7,Mid LAD,LCA,LAD,57.479554,-7.405083,178,169
6,8,Distal LAD,LCA,LAD,67.953616,-5.197806,577,518
7,10,Second diagonal (D2),LCA,LAD,46.281006,-22.942096,83,11
8,11,Proximal LCX,LCA,LCX,78.107470,-11.977179,316,316
9,13,Distal LCX,LCA,LCX,62.188656,-2.842070,406,318


<a id="step5"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">5. Segment Stenosis Severity Classification</p>

At this stage we classify **each anatomical segment**, not the whole patient. The ranges below follow the stenosis severity intervals used by CAD-RADS 2.0, but the output is only a **per-segment severity grade**.

- **0:** 0% — no visible stenosis.
- **1:** 1-24% — minimal stenosis.
- **2:** 25-49% — mild stenosis.
- **3:** 50-69% — moderate stenosis.
- **4:** 70-99% — severe stenosis.
- **5:** 100% — total occlusion.

The final patient-level CAD-RADS category will be computed later from these segment summaries plus the complete CAD-RADS reporting rules. Values are clipped to the physiologic interval `[0, 100]` before classification so small numerical artifacts do not create impossible categories.

In [13]:
# Purpose: assign per-segment stenosis severity grades from maximum segment %AS.
def classify_segment_stenosis_severity(max_pct_as: float) -> pd.Series:
    """Return segment-level stenosis severity grade and label for one maximum %AS."""
    if pd.isna(max_pct_as):
        return pd.Series(
            {"Stenosis_Severity_Grade": pd.NA, "Stenosis_Severity_Label": "Not assessable"}
        )

    value = float(np.clip(max_pct_as, 0, 100))
    if value == 0:
        grade, label = 0, "No visible stenosis"
    elif value < 25:
        grade, label = 1, "Minimal stenosis"
    elif value < 50:
        grade, label = 2, "Mild stenosis"
    elif value < 70:
        grade, label = 3, "Moderate stenosis"
    elif value < 100:
        grade, label = 4, "Severe stenosis"
    else:
        grade, label = 5, "Total occlusion"

    return pd.Series(
        {"Stenosis_Severity_Grade": grade, "Stenosis_Severity_Label": label}
    )


segment_summary[["Stenosis_Severity_Grade", "Stenosis_Severity_Label"]] = segment_summary[
    "Max_pct_AS"
].apply(classify_segment_stenosis_severity)
segment_summary["Max_pct_AS"] = segment_summary["Max_pct_AS"].clip(lower=0, upper=100)
segment_summary["Mean_pct_AS"] = segment_summary["Mean_pct_AS"].clip(lower=0, upper=100)

segment_summary = segment_summary.sort_values(
    ["Stenosis_Severity_Grade", "Max_pct_AS", "Segment_ID"], ascending=[False, False, True]
).reset_index(drop=True)

print("Segment-level stenosis severity summary:")
display(segment_summary)

Segment-level stenosis severity summary:


,Segment_ID,Segment_Name,Artery_Type,Specific_Artery,Max_pct_AS,Mean_pct_AS,Point_Count,Valid_pct_AS_Count,Stenosis_Severity_Grade,Stenosis_Severity_Label
0,19,Unmapped segment 19,Unmapped,Unmapped,90.142826,7.753647,471,195,4,Severe stenosis
1,6,Proximal LAD,LCA,LAD,83.205781,0.000000,129,129,4,Severe stenosis
2,11,Proximal LCX,LCA,LCX,78.107470,0.000000,316,316,4,Severe stenosis
3,17,LCX posterolateral branch,LCA,LCX,77.680517,0.000000,703,639,4,Severe stenosis
4,1,Proximal RCA,RCA,RCA,71.371532,25.231111,129,60,4,Severe stenosis
5,8,Distal LAD,LCA,LAD,67.953616,0.000000,577,518,3,Moderate stenosis
6,13,Distal LCX,LCA,LCX,62.188656,0.000000,406,318,3,Moderate stenosis
7,2,Mid RCA,RCA,RCA,59.746584,0.642120,349,349,3,Moderate stenosis
8,7,Mid LAD,LCA,LAD,57.479554,0.000000,178,169,3,Moderate stenosis
9,3,Distal RCA,RCA,RCA,54.120927,0.000000,411,286,3,Moderate stenosis


<a id="step6"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">6. Segment Summary Readout</p>

This section gives a compact readout of the segment table. It identifies the segment(s) with the highest stenosis severity grade and separately displays left main/proximal LAD findings because those territories have higher clinical consequence than many distal or small branch segments.

This readout is still **not a patient-level CAD-RADS score**; it is the structured segment evidence that will support that later computation.

In [14]:
# Purpose: print a compact segment-level stenosis readout for the sample.
assessable = segment_summary.dropna(subset=["Stenosis_Severity_Grade"]).copy()

if assessable.empty:
    print("No assessable segments were found.")
else:
    worst_grade = int(assessable["Stenosis_Severity_Grade"].max())
    worst_segments = assessable.loc[assessable["Stenosis_Severity_Grade"] == worst_grade]
    worst_names = ", ".join(worst_segments["Segment_Name"].tolist())

    print(f"Patient/sample                    : {SAMPLE_NAME}")
    print(f"Highest segment stenosis grade    : {worst_grade}")
    print(f"Most severe segment(s)            : {worst_names}")
    print("Note: this is not the final patient-level CAD-RADS score.")

    high_value_segments = assessable.loc[assessable["Segment_ID"].isin([5, 6])]
    if not high_value_segments.empty:
        print("\nHigh-consequence territory check:")
        for row in high_value_segments.itertuples(index=False):
            print(
                f"- {row.Segment_Name}: stenosis grade {row.Stenosis_Severity_Grade} "
                f"({row.Stenosis_Severity_Label}), max %AS={row.Max_pct_AS:.2f}"
            )

Patient/sample                    : Normal_2
Highest segment stenosis grade    : 4
Most severe segment(s)            : Unmapped segment 19, Proximal LAD, Proximal LCX, LCX posterolateral branch, Proximal RCA
Note: this is not the final patient-level CAD-RADS score.

High-consequence territory check:
- Proximal LAD: stenosis grade 4 (Severe stenosis), max %AS=83.21
- Left Main: stenosis grade 0 (No visible stenosis), max %AS=0.00


<a id="step7"></a>
# <p style="background-color:green; font-family:calibri; color:white; font-size:120%; text-align:center; border-radius:20px 20px;">7. Export Segment Summary</p>

The final table is exported as an Excel file for downstream reporting and manual review:

`results/block3_results/segment stenosis/{sample_name}/stenosis_summary_{sample_name}.xlsx`

The exported table keeps the anatomical dictionary fields, maximum stenosis ratio, and segment-level stenosis severity fields so it can later be used to compute the patient-level CAD-RADS score.

In [15]:
# Purpose: export the final segment-level stenosis summary.
output_dir = OUTPUT_ROOT / SAMPLE_NAME
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"stenosis_summary_{SAMPLE_NAME}.xlsx"

export_columns = [
    "Segment_ID",
    "Segment_Name",
    "Artery_Type",
    "Specific_Artery",
    "Max_pct_AS",
    "Stenosis_Severity_Grade",
    "Stenosis_Severity_Label",
    "Point_Count",
    "Valid_pct_AS_Count",
]

segment_summary[export_columns].to_excel(output_path, index=False)

print("Export complete:")
print(output_path)
display(segment_summary[export_columns])

Export complete:
C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\results\block3_results\segment stenosis\Normal_2\stenosis_summary_Normal_2.xlsx


,Segment_ID,Segment_Name,Artery_Type,Specific_Artery,Max_pct_AS,Stenosis_Severity_Grade,Stenosis_Severity_Label,Point_Count,Valid_pct_AS_Count
0,19,Unmapped segment 19,Unmapped,Unmapped,90.142826,4,Severe stenosis,471,195
1,6,Proximal LAD,LCA,LAD,83.205781,4,Severe stenosis,129,129
2,11,Proximal LCX,LCA,LCX,78.107470,4,Severe stenosis,316,316
3,17,LCX posterolateral branch,LCA,LCX,77.680517,4,Severe stenosis,703,639
4,1,Proximal RCA,RCA,RCA,71.371532,4,Severe stenosis,129,60
5,8,Distal LAD,LCA,LAD,67.953616,3,Moderate stenosis,577,518
6,13,Distal LCX,LCA,LCX,62.188656,3,Moderate stenosis,406,318
7,2,Mid RCA,RCA,RCA,59.746584,3,Moderate stenosis,349,349
8,7,Mid LAD,LCA,LAD,57.479554,3,Moderate stenosis,178,169
9,3,Distal RCA,RCA,RCA,54.120927,3,Moderate stenosis,411,286
